In [1]:
# ============================================================
# RETEST HYBRID RRF — dùng E1 từ docTR + BM25 (thay EasyOCR)
# Thay đổi duy nhất so với bản cũ: ưu tiên tìm artifact docTR
# ============================================================
import os, glob, json, zipfile, shutil
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm

WORK_ARTIFACT_DIR = "/kaggle/working/docvqa_colqwen2_artifacts"
RETRIEVAL_DIR     = f"{WORK_ARTIFACT_DIR}/retrieval_results"
os.makedirs(RETRIEVAL_DIR, exist_ok=True)

E0_RESULTS_PATH = f"{RETRIEVAL_DIR}/e0_dense_topk_results.json"
E1_RESULTS_PATH = f"{RETRIEVAL_DIR}/e1_bm25_topk_results.json"
e0_metrics_path = f"{RETRIEVAL_DIR}/e0_dense_metrics.json"
e1_metrics_path = f"{RETRIEVAL_DIR}/e1_bm25_metrics.json"


# ------------------------------------------------------------
# SETUP: ưu tiên artifact docTR (lớn hơn vì có thêm ocr_text_shards_doctr/)
# Nếu không tìm thấy docTR, fallback về bất kỳ artifact nào có E0
# ------------------------------------------------------------
def find_and_copy(target_path, filename_pattern):
    if os.path.exists(target_path):
        return True
    hits = glob.glob(f"/kaggle/input/**/{filename_pattern}", recursive=True)
    if hits:
        shutil.copy(hits[0], target_path)
        print(f"📄 Copied: {hits[0]}")
        return True
    return False


if not os.path.exists(E0_RESULTS_PATH):
    all_zips = [
        p for p in glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if "docvqa_colqwen2_artifacts" in os.path.basename(p)
    ]
    assert all_zips, "❌ Không tìm thấy artifact zip trong /kaggle/input"

    # ── SỬA CHÍNH: ưu tiên zip nào chứa ocr_text_shards_doctr (= docTR artifact) ──
    doctr_zips, other_zips = [], []
    for zp in all_zips:
        with zipfile.ZipFile(zp, "r") as zf:
            has_doctr = any("ocr_text_shards_doctr" in n for n in zf.namelist())
        (doctr_zips if has_doctr else other_zips).append(zp)

    if doctr_zips:
        zip_path = max(doctr_zips, key=os.path.getsize)
        print(f"📦 Dùng artifact docTR: {zip_path}")
    else:
        zip_path = max(other_zips, key=os.path.getsize)
        print(f"⚠️  Không tìm thấy artifact docTR — dùng artifact cũ nhất: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/kaggle/working")

assert find_and_copy(E0_RESULTS_PATH, "e0_dense_topk_results.json"), "❌ Thiếu E0"
assert find_and_copy(E1_RESULTS_PATH, "e1_bm25_topk_results.json"), "❌ Thiếu E1"
find_and_copy(e0_metrics_path, "e0_dense_metrics.json")
find_and_copy(e1_metrics_path, "e1_bm25_metrics.json")

# In ra OCR engine nào đang được dùng để xác nhận
with open(E1_METRICS_PATH := e1_metrics_path, encoding="utf-8") as f:
    e1_meta = json.load(f)
print(f"\n✅ E1 OCR engine: {e1_meta.get('ocr_engine', e1_meta.get('experiment', 'unknown'))}")


# ------------------------------------------------------------
# LOAD + VALIDATE
# ------------------------------------------------------------
with open(E0_RESULTS_PATH, "r", encoding="utf-8") as f:
    e0_results = json.load(f)
with open(E1_RESULTS_PATH, "r", encoding="utf-8") as f:
    e1_results = json.load(f)

assert len(e0_results) == len(e1_results), "❌ Số query E0/E1 không khớp"
mismatched = sum(
    1 for a, b in zip(e0_results, e1_results)
    if a["query_index"] != b["query_index"] or
       a["ground_truth_image_filename"] != b["ground_truth_image_filename"]
)
assert mismatched == 0, f"❌ {mismatched} dòng lệch thứ tự"
print(f"✅ E0 ({len(e0_results):,}) + E1 ({len(e1_results):,}) khớp thứ tự")


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
def compute_retrieval_metrics(results, k_values=[1, 5, 10]):
    metrics = {}
    n = len(results)
    for k in k_values:
        hit = sum(
            1 for item in results
            if item["ground_truth_image_filename"] in
               [x["image_id"] for x in item["topk"][:k]]
        )
        metrics[f"Recall@{k}"] = hit / n

    rr_list = []
    for item in results:
        gt = item["ground_truth_image_filename"]
        for rank, x in enumerate(item["topk"][:10], 1):
            if x["image_id"] == gt:
                rr_list.append(1.0 / rank); break
        else:
            rr_list.append(0.0)
    metrics["MRR@10"] = float(np.mean(rr_list))

    ndcg_list = []
    for item in results:
        gt = item["ground_truth_image_filename"]
        for rank, x in enumerate(item["topk"][:10], 1):
            if x["image_id"] == gt:
                ndcg_list.append(1.0 / np.log2(rank + 1)); break
        else:
            ndcg_list.append(0.0)
    metrics["NDCG@10"] = float(np.mean(ndcg_list))
    return metrics


# ------------------------------------------------------------
# E2 RRF FUSION — giữ nguyên logic
# ------------------------------------------------------------
K_RRF = 60
TOP_K = 10

E2_RESULTS_PATH = f"{RETRIEVAL_DIR}/e2_rrf_topk_results.json"
E2_METRICS_PATH = f"{RETRIEVAL_DIR}/e2_rrf_metrics.json"


def rrf_fuse_single(e0_topk, e1_topk, k_rrf=K_RRF, top_k=TOP_K):
    scores = {}
    for item in e0_topk:
        img_id = item["image_id"]
        scores[img_id] = scores.get(img_id, 0.0) + 1.0 / (k_rrf + item["rank"])
    for item in e1_topk:
        img_id = item["image_id"]
        scores[img_id] = scores.get(img_id, 0.0) + 1.0 / (k_rrf + item["rank"])
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [
        {"rank": r + 1, "image_id": img_id, "score": float(score)}
        for r, (img_id, score) in enumerate(sorted_items)
    ]


results = []
for e0_item, e1_item in zip(tqdm(e0_results, desc="E2 RRF (docTR)"), e1_results):
    results.append({
        "query_index": e0_item["query_index"],
        "query": e0_item["query"],
        "ground_truth_image_filename": e0_item["ground_truth_image_filename"],
        "answer": e0_item["answer"],
        "topk": rrf_fuse_single(e0_item["topk"], e1_item["topk"]),
    })

metrics = compute_retrieval_metrics(results, k_values=[1, 5, 10])
metrics_info = {
    "experiment": "E2_RRF_Fusion_Dense_BM25_docTR",   # ghi rõ engine
    "ocr_engine": "doctr",
    "k_rrf": K_RRF, "top_k": TOP_K,
    "num_queries": len(results),
    "created_at": datetime.now().isoformat(),
    "metrics": metrics,
}

with open(E2_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(E2_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_info, f, ensure_ascii=False, indent=2)

print("\n✅ E2 RRF (docTR) finished!")
for k, v in metrics.items():
    print(f"   {k}: {v:.4f}")


# ------------------------------------------------------------
# BẢNG SO SÁNH E0 / E1(docTR) / E2(docTR)
# ------------------------------------------------------------
with open(e0_metrics_path, encoding="utf-8") as f:
    e0_m = json.load(f)["metrics"]
with open(e1_metrics_path, encoding="utf-8") as f:
    e1_m = json.load(f)["metrics"]

print(f"\n{'Metric':<12} {'E0 Dense':>10} {'E1 docTR':>10} {'E2 RRF':>10}   Delta(E2-E0)")
print("-" * 60)
for k in ["Recall@1", "Recall@5", "Recall@10", "MRR@10", "NDCG@10"]:
    v0, v1, v2 = e0_m[k], e1_m[k], metrics[k]
    delta = v2 - v0
    sign = "+" if delta >= 0 else ""
    print(f"{k:<12} {v0:>10.4f} {v1:>10.4f} {v2:>10.4f}   {sign}{delta:.4f}")
print("=" * 60)


# ------------------------------------------------------------
# ZIP LẠI
# ------------------------------------------------------------
zip_path = "/kaggle/working/docvqa_colqwen2_artifacts.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive("/kaggle/working/docvqa_colqwen2_artifacts", "zip",
                    "/kaggle/working", "docvqa_colqwen2_artifacts")
print(f"\n✅ Đã zip: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    for fname in ["e0_dense_metrics.json", "e1_bm25_metrics.json",
                  "e2_rrf_topk_results.json", "e2_rrf_metrics.json"]:
        found = any(n.endswith(fname) for n in names)
        print(f"{'✅' if found else '❌'} {fname}")

📦 Dùng artifact docTR: /kaggle/input/notebooks/hnganhdng/bm25-doctr/docvqa_colqwen2_artifacts.zip

✅ E1 OCR engine: doctr (det_arch=db_resnet50, reco_arch=parseq)
✅ E0 (2,000) + E1 (2,000) khớp thứ tự


E2 RRF (docTR):   0%|          | 0/2000 [00:00<?, ?it/s]


✅ E2 RRF (docTR) finished!
   Recall@1: 0.3385
   Recall@5: 0.5130
   Recall@10: 0.5725
   MRR@10: 0.4088
   NDCG@10: 0.4479

Metric         E0 Dense   E1 docTR     E2 RRF   Delta(E2-E0)
------------------------------------------------------------
Recall@1         0.3775     0.1555     0.3385   -0.0390
Recall@5         0.5630     0.2375     0.5130   -0.0500
Recall@10        0.6295     0.2705     0.5725   -0.0570
MRR@10           0.4580     0.1907     0.4088   -0.0492
NDCG@10          0.4992     0.2097     0.4479   -0.0513

✅ Đã zip: /kaggle/working/docvqa_colqwen2_artifacts.zip (1497.7 MB)
✅ e0_dense_metrics.json
✅ e1_bm25_metrics.json
✅ e2_rrf_topk_results.json
✅ e2_rrf_metrics.json
